# Image Detection Demo with PytorchWildlife

This tutorial guides you on how to use PyTorchWildlife to separate positive and negative animal detections. We will go through the process of setting up the environment, defining the detection model, as well as performing inference and saving the results in different ways.

## Prerequisites
Install PytorchWildlife running the following commands:
```bash
conda create -n pytorch_wildlife python=3.8 -y
conda activate pytorch_wildlife
pip install PytorchWildlife
```
Also, make sure you have a CUDA-capable GPU if you intend to run the model on a GPU. This notebook can also run on CPU.

## Importing libraries
First, we'll start by importing the necessary libraries and modules.

In [2]:
import sys, os
print("sys.executable:", sys.executable)
print("LD_LIBRARY_PATH:", os.environ.get("LD_LIBRARY_PATH", ""))

sys.executable: /home/mo/anaconda3/envs/pw20251118/bin/python
LD_LIBRARY_PATH: /usr/local/cuda-12.1/lib64:/usr/local/cuda-12.1/lib64


In [3]:
import os, sys

extra = "/usr/local/lib/python3.10/dist-packages/nvidia/nvjitlink/lib"
os.environ["LD_LIBRARY_PATH"] = extra + ":" + os.environ.get("LD_LIBRARY_PATH", "")

print("sys.executable:", sys.executable)
print("LD_LIBRARY_PATH:", os.environ["LD_LIBRARY_PATH"])

import ctypes
ctypes.CDLL("libnvJitLink.so.12")
print("Loaded libnvJitLink.so.12 OK")

sys.executable: /home/mo/anaconda3/envs/pw20251118/bin/python
LD_LIBRARY_PATH: /usr/local/lib/python3.10/dist-packages/nvidia/nvjitlink/lib:/usr/local/cuda-12.1/lib64:/usr/local/cuda-12.1/lib64
Loaded libnvJitLink.so.12 OK


In [4]:
#%pip install PytorchWildlife
import os, sys
import importlib, inspect
from pathlib import Path

# Prefer the local repo copy of PytorchWildlife (so edits in this workspace take effect).
def _find_repo_root(start_dir: str) -> str | None:
    p = Path(start_dir).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "PytorchWildlife").is_dir() and (candidate / "setup.py").is_file():
            return str(candidate)
    return None

# Try to locate the repo root from the current working directory; fall back to the known workspace path.
repo_root = _find_repo_root(os.getcwd()) or "/home/mo/CameraTraps"
local_pkg_dir = os.path.join(repo_root, "PytorchWildlife")
if not os.path.isdir(local_pkg_dir):
    raise RuntimeError(f"Local PytorchWildlife not found at {local_pkg_dir}")

# Ensure local repo path wins, and purge any already-imported installed package modules.
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
for m in list(sys.modules.keys()):
    if m == "PytorchWildlife" or m.startswith("PytorchWildlife."):
        del sys.modules[m]

import torch
from PytorchWildlife.models import detection as pw_detection
from PytorchWildlife import utils as pw_utils
import PytorchWildlife.utils.post_process as pw_post_process

# Reload to pick up any local edits
importlib.reload(pw_post_process)
importlib.reload(pw_utils)

print("PytorchWildlife loaded from:", os.path.dirname(pw_post_process.__file__))
print("post_process.detection_folder_separation:", inspect.signature(pw_post_process.detection_folder_separation))
print("pw_utils.detection_folder_separation:", inspect.signature(pw_utils.detection_folder_separation))

PytorchWildlife loaded from: /home/mo/CameraTraps/PytorchWildlife/utils
post_process.detection_folder_separation: (json_file, img_path, destination_path, confidence_threshold, output_subdir=None, copy_mode='both', preserve_relative_paths=False)
pw_utils.detection_folder_separation: (json_file, img_path, destination_path, confidence_threshold, output_subdir=None, copy_mode='both', preserve_relative_paths=False)


## Model Initialization
We will initialize the MegaDetectorV5 model for image detection. This model is designed for detecting animals in images.

In [5]:
# Ask user which device to use: GPU or CPU
choice = input("Select device [gpu/cpu] (default: gpu): " ).strip().lower()

# ---- CPU OPTION (COMMENTED OUT BY DEFAULT) ----
# if choice in ("cpu", "c"):
#     DEVICE = "cpu"
#     print("Using CPU.")
# else:

# ---- GPU OPTION (DEFAULT) ----
if choice in ("cpu", "c"):
    print("CPU option is currently disabled in this notebook. Using GPU instead.")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Please enable GPU or uncomment the CPU option.")

# Show available GPU indices
num_gpus = torch.cuda.device_count()
print(f"Number of available GPUs: {num_gpus}")
for idx in range(num_gpus):
    print(f"GPU {idx}: {torch.cuda.get_device_name(idx)}")

# Ask user which GPU index to use (default 0)
gpu_idx_input = input(f"Select GPU index [0-{num_gpus-1}] (default: 0): " ).strip()
if gpu_idx_input == "":
    gpu_idx = 0
else:
    gpu_idx = int(gpu_idx_input)
    if gpu_idx < 0 or gpu_idx >= num_gpus:
        raise ValueError(f"Invalid GPU index {gpu_idx}. Must be between 0 and {num_gpus-1}.")

DEVICE = "cuda"
torch.cuda.set_device(gpu_idx)
print(f"Using GPU: cuda:{gpu_idx}")

# Initializing the MegaDetectorV6 or V5 model for image detection
# Valid versions are MDV6-yolov9-c, MDV6-yolov9-e, MDV6-yolov10-c, MDV6-yolov10-e or MDV6-rtdetr-c
#detection_model = pw_detection.MegaDetectorV6(device=DEVICE, pretrained=True, version="MDV6-yolov10-e")

# Uncomment the following line to use MegaDetectorV5 instead of MegaDetectorV6
detection_model = pw_detection.MegaDetectorV5(device=DEVICE, pretrained=True, version="a")

Number of available GPUs: 2
GPU 0: NVIDIA GeForce RTX 4090
GPU 1: NVIDIA GeForce RTX 4090
Using GPU: cuda:1


Fusing layers... 
Model summary: 733 layers, 140054656 parameters, 0 gradients, 208.8 GFLOPs


## Variable definition
In order to process the batch detection, we will define an input directory where the images are stored, a confidence threshold and an output directory to copy the positive and negative images into distinctive folders. If you want to follow this tutorial with your own data, modify the following variables.

In [6]:
#tgt_folder_path = os.path.join(".","demo_data","imgs")
#output_path = "folder_separation"
tgt_folder_path = "/media/mo/nvme0n1/MD_up_TT/HG"
#output_path = "/home/mo/HDD1/MD_up_TT_output/MG_MDV6-yolov10-e"
#tgt_folder_path = "/mnt/sshfs_s3it/pec_filestorage/CamTrapData_General/CAM002"
#tgt_folder_path = "/home/mo/mnt/nextcloud/CamTrapData_General/CAM002"
#tgt_folder_path = "/mnt/sshfs_s3it/pec_filestorage/CamTrapData_General/CAM001/20240827"
output_path = "/media/mo/nvme0n1/20260112_MDv5a_CleanData"
threshold = 0.1 #default is 0.2

In [11]:
# Advanced options
MULTI_GPU = True  # True = use multiple GPUs with multiprocessing
GPU_IDS = [0, 1]  # GPUs to use when MULTI_GPU=True
MP_START_METHOD = "spawn"  # CUDA multiprocessing requires 'spawn'
MAX_PARALLEL_PROCESSES = None  # None = len(GPU_IDS)
MODEL_FAMILY = "MegaDetectorV5"  # "MegaDetectorV5" or "MegaDetectorV6"
MODEL_VERSION = "a"  # V5: "a" | V6: "MDV6-yolov10-e", etc.
AUTO_BATCH_SHRINK = True  # shrink batch size on CUDA OOM
OOM_RETRY_MIN_BATCH = 1
SKIP_EMPTY_FOLDERS = True
DRY_RUN = False  # True = count only, no detection/separation

# Multi-GPU logging/progress controls
SHOW_PATHS_MULTI_GPU = True  # True shows per-image paths (may interleave output) in log files
CAPTURE_WORKER_STDOUT = True  # True writes per-GPU logs to file (recommended)
WORKER_LOG_DIR = ""  # empty = output_root/logs
HEARTBEAT_SECONDS = 10  # write current file path every N seconds
HEARTBEAT_ENABLED = False  # True = enable heartbeat logging

## Batch detection + folder separation (interactive, per-subfolder)
This cell lists subfolders under the site root (alphabetical), asks where to start and what to copy, then processes folder-by-folder with cleanup between folders.

In [12]:
import os, time, gc, sys, subprocess, shutil, inspect
from pathlib import Path
import multiprocessing as mp

import torch
from tqdm.auto import tqdm

# Ensure child processes can import local repo modules
repo_root = globals().get("repo_root", "/home/mo/CameraTraps")
py_path = os.environ.get("PYTHONPATH", "")
if repo_root not in py_path.split(":" ):
    os.environ["PYTHONPATH"] = f"{repo_root}:{py_path}" if py_path else repo_root

from PytorchWildlife.utils.multi_gpu_worker import worker_loop

# Uses existing variables from earlier cells:
# - tgt_folder_path
# - output_path
# - threshold
# - detection_model (single-GPU path only)
# - pw_utils

def _log(msg: str) -> None:
    print(msg, flush=True)
    sys.stdout.flush()

def _cuda_stats_all() -> str:
    if not torch.cuda.is_available():
        return "CUDA not available"
    lines = []
    for device in range(torch.cuda.device_count()):
        try:
            props = torch.cuda.get_device_properties(device)
            total_gb = props.total_memory / (1024**3)
            free_b, total_b = torch.cuda.mem_get_info(device)
            free_gb = free_b / (1024**3)
            used_gb = (total_b - free_b) / (1024**3)
        except Exception:
            total_gb = None
            free_gb = None
            used_gb = None

        alloc_gb = torch.cuda.memory_allocated(device) / (1024**3)
        reserv_gb = torch.cuda.memory_reserved(device) / (1024**3)
        if total_gb is not None:
            lines.append(
                f"cuda:{device} used={used_gb:.2f}GB free={free_gb:.2f}GB total={total_gb:.2f}GB | "
                f"alloc={alloc_gb:.2f}GB reserved={reserv_gb:.2f}GB"
            )
        else:
            lines.append(f"cuda:{device} alloc={alloc_gb:.2f}GB reserved={reserv_gb:.2f}GB")
    return "\n".join(lines)

def _nvidia_smi_snapshot() -> str:
    if shutil.which("nvidia-smi") is None:
        return "nvidia-smi not found"
    cmd = ["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total", "--format=csv,noheader,nounits"]
    try:
        out = subprocess.check_output(cmd, text=True).strip()
        lines = [f"nvidia-smi: {line.strip()}" for line in out.splitlines() if line.strip()]
        return "\n".join(lines) if lines else "nvidia-smi returned no data"
    except Exception as exc:
        return f"nvidia-smi failed: {exc}"

# ---- Resolve site_root + site_name (supports selecting a date folder like YYYYMMDD) ----
tgt_folder_path = os.path.normpath(tgt_folder_path)
target_folder_name = os.path.basename(tgt_folder_path)

default_start_folder = ""
if target_folder_name.isdigit() and len(target_folder_name) == 8:
    site_root = os.path.dirname(tgt_folder_path)
    site_name = os.path.basename(site_root)
    default_start_folder = target_folder_name
else:
    site_root = tgt_folder_path
    site_name = target_folder_name

output_root = os.path.join(output_path, site_name)
os.makedirs(output_root, exist_ok=True)

_log(f"\nSite root:   {site_root}")
_log(f"Site name:   {site_name}")
_log(f"Output root: {output_root}\n")
_log(_cuda_stats_all())
_log(_nvidia_smi_snapshot())

# ---- List immediate subfolders (alphabetical) ----
subfolders = sorted([p for p in Path(site_root).iterdir() if p.is_dir()], key=lambda p: p.name.lower())
if subfolders:
    _log("Folders found (alphabetical):")
    for i, p in enumerate(subfolders):
        _log(f"  [{i:03d}] {p.name}")
else:
    _log("No subfolders found; will process images directly under site_root.")

# ---- Ask once: start folder + copy mode (+ optional overrides) ----
start_in = input(f"\nStart from which folder? (name or index; empty = {default_start_folder or 'first'}): ").strip()
if start_in == "" and default_start_folder:
    start_in = default_start_folder

copy_mode = input("Copy which images? [both/animal/no_animal] (default: both): ").strip().lower() or "both"
if copy_mode not in ("both", "animal", "no_animal"):
    raise ValueError("Invalid choice. Use 'both', 'animal', or 'no_animal'.")

resume_in = input("Resume mode? Skip folders with existing per-folder JSON? [y/n] (default: y): ").strip().lower() or "y"
resume_skip = (resume_in == "y")

batch_in = input("Batch size override? (empty = 30): ").strip()
batch_size = int(batch_in) if batch_in else 30

det_conf_in = input(f"Inference det_conf_thres? (empty = {threshold}): ").strip()
det_conf_thres = float(det_conf_in) if det_conf_in else float(threshold)

path_log_every = input("Log inference path every N images (default 25): ").strip()
path_log_every = int(path_log_every) if path_log_every else 25
path_log_mode = input("Inference path display mode [line/tqdm] (default: line): ").strip().lower() or "line"

_log("\nRun selections:")
_log(f"  start_from='{start_in or default_start_folder or 'first'}'")
_log(f"  copy_mode='{copy_mode}'")
_log(f"  resume_skip={resume_skip}")
_log(f"  batch_size={batch_size}")
_log(f"  det_conf_thres={det_conf_thres}")
_log(f"  path_log_every={path_log_every}")
_log(f"  path_log_mode='{path_log_mode}'")
_log(f"  multi_gpu={MULTI_GPU} gpu_ids={GPU_IDS}")
_log(f"  model_family={MODEL_FAMILY} model_version={MODEL_VERSION}")
_log(f"  capture_worker_stdout={CAPTURE_WORKER_STDOUT} show_paths_multi_gpu={SHOW_PATHS_MULTI_GPU}")
_log(f"  heartbeat_enabled={HEARTBEAT_ENABLED} heartbeat_seconds={HEARTBEAT_SECONDS}")

def _start_index_from_input(items, s: str) -> int:
    if not items:
        return 0
    if s == "":
        return 0
    if s.isdigit():
        idx = int(s)
        if idx < 0 or idx >= len(items):
            raise ValueError(f"Invalid index {idx}; expected 0..{len(items)-1}")
        return idx
    matches = [i for i, p in enumerate(items) if p.name == s]
    if not matches:
        raise ValueError(f"Folder '{s}' not found under {site_root}")
    return matches[0]

def _cleanup_between_folders() -> None:
    gc.collect()
    if torch.cuda.is_available():
        try:
            torch.cuda.synchronize()
        except Exception:
            pass
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass
        try:
            torch.cuda.reset_peak_memory_stats()
        except Exception:
            pass

def _count_images(folder_path: str) -> int:
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".gif", ".webp"}
    count = 0
    for root, _, files in os.walk(folder_path):
        for f in files:
            if os.path.splitext(f)[1].lower() in exts:
                count += 1
    return count

def _is_oom(exc: BaseException) -> bool:
    if isinstance(exc, torch.cuda.OutOfMemoryError):
        return True
    msg = str(exc).lower()
    return "out of memory" in msg or ("cuda" in msg and "memory" in msg)

def _run_batch_detection_single(folder_path: str, bs: int):
    cur_bs = bs
    while True:
        try:
            if "show_paths" in inspect.signature(detection_model.batch_image_detection).parameters:
                return detection_model.batch_image_detection(
                    folder_path,
                    batch_size=cur_bs,
                    det_conf_thres=det_conf_thres,
                    id_strip=site_root,
                    show_paths=True,
                    path_log_every=path_log_every,
                    path_log_mode=path_log_mode,
                ), cur_bs
            return detection_model.batch_image_detection(
                folder_path,
                batch_size=cur_bs,
                det_conf_thres=det_conf_thres,
                id_strip=site_root,
            ), cur_bs
        except Exception as exc:
            if not (AUTO_BATCH_SHRINK and _is_oom(exc) and cur_bs > OOM_RETRY_MIN_BATCH):
                raise
            new_bs = max(OOM_RETRY_MIN_BATCH, cur_bs // 2)
            _log(f"OOM at batch_size={cur_bs}; retrying with batch_size={new_bs}")
            cur_bs = new_bs
            _cleanup_between_folders()

# ---- Build folder list ----
start_idx = _start_index_from_input(subfolders, start_in)
folders_to_process = subfolders[start_idx:] if subfolders else [Path(site_root)]

_log(f"\nWill process {len(folders_to_process)} folder(s), starting at index {start_idx}.")
_log(f"copy_mode='{copy_mode}' | det_conf_thres={det_conf_thres} | separation threshold={float(threshold)} | batch_size={batch_size}\n")

if MULTI_GPU:
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available; MULTI_GPU requires GPUs.")
    if not GPU_IDS:
        raise ValueError("GPU_IDS is empty.")

    max_procs = MAX_PARALLEL_PROCESSES or len(GPU_IDS)
    worker_gpu_ids = GPU_IDS[:max_procs]
    available_methods = mp.get_all_start_methods()
    if MP_START_METHOD not in available_methods:
        fallback = "spawn" if "spawn" in available_methods else available_methods[0]
        _log(f"Start method '{MP_START_METHOD}' not available; falling back to '{fallback}'.")
        MP_START_METHOD = fallback
    ctx = mp.get_context(MP_START_METHOD)
    task_queue = ctx.Queue()
    result_queue = ctx.Queue()

    cfg = {
        "site_root": site_root,
        "site_name": site_name,
        "output_root": output_root,
        "output_path": output_path,
        "threshold": threshold,
        "copy_mode": copy_mode,
        "batch_size": batch_size,
        "det_conf_thres": det_conf_thres,
        "path_log_every": path_log_every,
        "path_log_mode": path_log_mode,
        "resume_skip": resume_skip,
        "skip_empty": SKIP_EMPTY_FOLDERS,
        "dry_run": DRY_RUN,
        "auto_batch_shrink": AUTO_BATCH_SHRINK,
        "oom_retry_min_batch": OOM_RETRY_MIN_BATCH,
        "model_family": MODEL_FAMILY,
        "model_version": MODEL_VERSION,
        "show_paths": SHOW_PATHS_MULTI_GPU,
        "capture_worker_stdout": CAPTURE_WORKER_STDOUT,
        "worker_log_dir": WORKER_LOG_DIR or None,
        "heartbeat_seconds": HEARTBEAT_SECONDS if HEARTBEAT_ENABLED else 0,
    }

    for folder in folders_to_process:
        task_queue.put(str(folder))
    for _ in worker_gpu_ids:
        task_queue.put(None)

    procs = []
    for gpu_id in worker_gpu_ids:
        p = ctx.Process(
            target=worker_loop,
            args=(gpu_id, task_queue, result_queue, cfg),
        )
        p.start()
        procs.append(p)

    total = len(folders_to_process)
    done = 0
    with tqdm(total=total, desc="Folders (multi-GPU)", unit="folder", dynamic_ncols=True) as pbar:
        while done < total:
            result = result_queue.get()
            status = result.get("status")
            if status == "worker_log":
                _log(f"GPU{result['gpu_id']} log: {result['log_path']}")
                continue
            if status == "ok":
                _log(f"✓ [cuda:{result['gpu_id']}] {result['folder']}")
                _log(result.get("message", ""))
                _log(f"  batch_size={result.get('batch_size')} seconds={result.get('seconds')}")
            elif status == "skipped_existing_json":
                _log(f"SKIP [cuda:{result['gpu_id']}] JSON exists: {result['json_file']}")
            elif status == "skipped_empty":
                _log(f"SKIP [cuda:{result['gpu_id']}] empty folder: {result['folder']}")
            elif status == "dry_run":
                _log(f"DRY_RUN [cuda:{result['gpu_id']}] {result['folder']} ({result.get('images')} images)")
            else:
                _log(f"ERROR [cuda:{result.get('gpu_id')}] {result.get('folder')}: {result.get('error')}")
            done += 1
            pbar.update(1)

    for p in procs:
        p.join()

    _log("\nAll done (multi-GPU).")

else:
    # ---- Single-GPU / single-process path ----
    for folder in tqdm(
        folders_to_process,
        desc=f"Folders @ {site_name}",
        unit="folder",
        dynamic_ncols=True,
        mininterval=0.5,
        smoothing=0.1,
        leave=True,
        position=0,
        ascii=False,
        file=sys.stdout,
        ncols=120,
    ):
        folder_path = str(folder)
        folder_leaf = os.path.basename(os.path.normpath(folder_path))
        json_file = os.path.join(output_root, f"detection_results__{folder_leaf}.json")

        if resume_skip and os.path.isfile(json_file):
            _log(f"\nSKIP (JSON exists): {json_file}")
            continue

        _log(f"\n=== Processing: {folder_path} ===")
        img_count = _count_images(folder_path)
        _log(f"Images in folder: {img_count}")
        if SKIP_EMPTY_FOLDERS and img_count == 0:
            _log("Skipping empty folder")
            continue

        _log(_cuda_stats_all())
        _log(_nvidia_smi_snapshot())
        t0 = time.time()

        if DRY_RUN:
            _log("DRY_RUN=True: skipping detection/separation")
            continue

        results, used_bs = _run_batch_detection_single(folder_path, batch_size)

        pw_utils.save_detection_json(
            results,
            json_file,
            categories=detection_model.CLASS_NAMES,
            exclude_category_ids=[],
            exclude_file_path=site_root,
        )
        _log(f"Saved JSON: {json_file}")

        sep_msg = pw_utils.detection_folder_separation(
            json_file,
            site_root,
            output_path,
            float(threshold),
            output_subdir=site_name,
            copy_mode=copy_mode,
            preserve_relative_paths=True,
        )
        _log(sep_msg)

        # Free RAM/GPU before the next folder
        del results
        _cleanup_between_folders()

        dt = time.time() - t0
        if torch.cuda.is_available():
            _log(_cuda_stats_all())
            _log(_nvidia_smi_snapshot())
            _log(f"Done: {folder_path} in {dt/60:.1f} min")
        else:
            _log(f"Done: {folder_path} in {dt/60:.1f} min")

    _log("\nAll done.")


Site root:   /media/mo/nvme0n1/MD_up_TT/HG
Site name:   HG
Output root: /media/mo/nvme0n1/20260112_MDv5a_CleanData/HG

cuda:0 used=1.37GB free=22.25GB total=23.62GB | alloc=0.00GB reserved=0.00GB
cuda:1 used=1.63GB free=22.01GB total=23.64GB | alloc=1.06GB reserved=1.17GB


nvidia-smi: 0, NVIDIA GeForce RTX 4090, 1402, 24564
nvidia-smi: 1, NVIDIA GeForce RTX 4090, 1670, 24564
Folders found (alphabetical):
  [000] O15
  [001] O16
  [002] O17
  [003] O18
  [004] P15
  [005] P16
  [006] P17
  [007] P18
  [008] Q15
  [009] Q16
  [010] Q17
  [011] Q18
  [012] R15
  [013] R16
  [014] R17
  [015] R18
  [016] S15
  [017] S16
  [018] S17
  [019] S18

Run selections:
  start_from='17'
  copy_mode='animal'
  resume_skip=False
  batch_size=25
  det_conf_thres=0.1
  path_log_every=25
  path_log_mode='line'
  multi_gpu=True gpu_ids=[0, 1]
  model_family=MegaDetectorV5 model_version=a
  capture_worker_stdout=True show_paths_multi_gpu=True
  heartbeat_enabled=False heartbeat_seconds=10

Will process 3 folder(s), starting at index 17.
copy_mode='animal' | det_conf_thres=0.1 | separation threshold=0.1 | batch_size=25



Folders (multi-GPU):   0%|          | 0/3 [00:00<?, ?folder/s]UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


GPU0 log: /media/mo/nvme0n1/20260112_MDv5a_CleanData/HG/logs/gpu0_pid1033525_20260126_124335.log
GPU1 log: /media/mo/nvme0n1/20260112_MDv5a_CleanData/HG/logs/gpu1_pid1033526_20260126_124335.log


Fusing layers... 
Fusing layers... 
Model summary: 733 layers, 140054656 parameters, 0 gradients, 208.8 GFLOPs
Model summary: 733 layers, 140054656 parameters, 0 gradients, 208.8 GFLOPs


...

✓ [cuda:0] /media/mo/nvme0n1/MD_up_TT/HG/S16
6792 images processed, 6791 files copied
  batch_size=25 seconds=418.28


Folders (multi-GPU):  67%|██████▋   | 2/3 [07:01<03:25, 205.36s/folder]

✓ [cuda:1] /media/mo/nvme0n1/MD_up_TT/HG/S18
13429 images processed, 13429 files copied
  batch_size=25 seconds=677.42


Folders (multi-GPU): 100%|██████████| 3/3 [15:19<00:00, 306.40s/folder]



All done (multi-GPU).


### Copyright (c) Microsoft Corporation. All rights reserved.
### Licensed under the MIT License.